## Imports

In [ ]:
import ast
import hashlib
import json
import os
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tqdm import tqdm

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
ANALYSIS_RESULTS_DIR = PROJECT_ROOT / "analysis" / "results"
SPEECH_SEMANTICS_DIR = DATA_DIR / "speech_semantics"
SPEECH_SEMANTICS_RESULTS_DIR = ANALYSIS_RESULTS_DIR / "speech_semantics"

## Task Performance and Behavioral Transfer
### Group 1 = Same Type; Group 2 = Different Type

In [ ]:
trials_df = pd.read_csv(
    DATA_DIR / "performance" / "trials_df.csv",
    converters={
        "true_output": ast.literal_eval,
        "submission_attempts": ast.literal_eval,
    },
)

trials_df["participant_id"] = trials_df["participant_id"].astype(str)
trials_df["trial_number"] = pd.to_numeric(trials_df["trial_number"], errors="coerce").astype(int)
trials_df["group"] = pd.to_numeric(trials_df["group"], errors="coerce").astype(int)
trials_df["rt"] = pd.to_numeric(trials_df["rt"], errors="coerce")
trials_df["rt_seconds"] = pd.to_numeric(trials_df["rt_seconds"], errors="coerce")
trials_df["is_correct"] = trials_df["is_correct"].astype(bool)

if "group_label" not in trials_df.columns:
    trials_df["group_label"] = trials_df["group"].map({
        1: "Same Type (Group 1)",
        2: "Different Types (Group 2)",
    })

print("number of trials:", len(trials_df))
print("number of participants:", trials_df["participant_id"].nunique())

### Response Times Across Trials

In [ ]:
correct_df = trials_df.loc[trials_df["is_correct"]].copy()
correct_df["rt_seconds"] = pd.to_numeric(correct_df["rt"], errors="coerce") / 1000

print("Correct trials:", len(correct_df))
print("Mean RT on correct trials (s):", round(correct_df["rt_seconds"].mean(), 2))
print("SD RT on correct trials (s):", round(correct_df["rt_seconds"].std(), 2))

### Accuracy Across Trials

In [ ]:
# --- Overall Accuracy ---
print("\nPerformance Statistics:")
print(f"Number of correct trials: {trials_df['is_correct'].sum()}")
print(f"Overall accuracy: {trials_df['is_correct'].mean() * 100:.2f}%")

# --- Accuracy by Task Type ---
print("\nPerformance by Task Type:")
task_type_stats = trials_df.groupby('task_type')['is_correct'].agg(['count', 'sum', 'mean'])
task_type_stats['accuracy'] = task_type_stats['mean'] * 100
print(task_type_stats)

# --- Accuracy by Group ---
print("\nPerformance by Group:")
group_stats = trials_df.groupby('group')['is_correct'].agg(['count', 'sum', 'mean'])
group_stats['accuracy'] = group_stats['mean'] * 100
print(group_stats)

# --- Accuracy by Task Type and Group ---
print("\nPerformance by Task Type and Group:")
task_group_stats = trials_df.groupby(['task_type', 'group'])['is_correct'].agg(['count', 'sum', 'mean'])
task_group_stats['accuracy'] = task_group_stats['mean'] * 100
print(task_group_stats.reset_index())

### Performance After First Success

In [ ]:
# 1) First correct trial per participant
first_success = (
    trials_df.loc[trials_df["is_correct"] == True]
    .sort_values(["participant_id", "trial_number"])
    .groupby("participant_id")["trial_number"]
    .first()
)

# Attach it
df = trials_df.copy()
df["first_success_trial"] = df["participant_id"].map(first_success)

# 2) Keep only participants with >=1 correct AND first correct not on trial 5
eligible_pids = first_success[(first_success >= 1) & (first_success <= 4)].index
df = df[df["participant_id"].isin(eligible_pids)].copy()

# 3) Keep only trials AFTER first success (strictly after)
df["post_first_success"] = df["trial_number"] > df["first_success_trial"]
post_df = df[df["post_first_success"]].copy()

# Sanity check: participants should have between 1 and 4 post trials depending on first_success_trial
post_counts = post_df.groupby("participant_id")["trial_number"].nunique()
print("Eligible participants:", len(eligible_pids))
print("Mean #post trials per eligible participant:", post_counts.mean())

# 4) Participant-level post-success accuracy, then group summary
post_acc_by_pid = (
    post_df.groupby(["participant_id", "group"])["is_correct"]
    .mean()
    .reset_index(name="post_accuracy")
)

summary = (
    post_acc_by_pid.groupby("group")["post_accuracy"]
    .agg(mean="mean", sem="sem", n="count")
    .reset_index()
)

print(summary)

In [ ]:
post_rt_by_pid = (
    post_df.groupby(["participant_id", "group"])["rt_seconds"]
    .mean()
    .reset_index(name="post_rt_seconds")
)

rt_summary = (
    post_rt_by_pid.groupby("group")["post_rt_seconds"]
    .agg(mean="mean", sem="sem", n="count")
    .reset_index()
)

print(rt_summary)

### Accuracy Across Trials by Problem Type

In [ ]:
# PARTICIPANT-LEVEL BOOTSTRAP (cluster bootstrap)
# -----------------------------
# Output folder
# -----------------------------
FIG_DIR = FIGURES_DIR / "performance"
os.makedirs(FIG_DIR, exist_ok=True)

# -----------------------------
# Styling
# -----------------------------
FIGSIZE = (7.8, 4.8)      # tighter x-axis width (was 9,5) but not too tight
DPI = 600
MARKERSIZE = 8
ELINEWIDTH = 2.2
CAPSIZE = 0               # <-- NO HATS
TITLE_FONTSIZE = 18
LABEL_FONTSIZE = 16
TICK_FONTSIZE  = 14
LEGEND_FONTSIZE = 13

GROUP_ALPHA = 0.85        # a bit higher since Different is black

OFFSETS = {
    1: -0.12,   # Same group
    2:  0.12,   # Different group
}

group_labels = {1: "Same Type (Group 1)", 2: "Different Types (Group 2)"}
group_colors = {1: "red", 2: "black"}     # <-- Different is black

# -----------------------------
# Data prep
# -----------------------------
df = trials_df.copy()
if "rt_seconds" not in df.columns:
    df["rt_seconds"] = df["rt"] / 1000

# -----------------------------
# Helper: participant-level bootstrap CI for mean
# -----------------------------
def pid_boot_ci95(df_sub, value_col, pid_col="participant_id", n_boot=10000, seed=0):
    pid_means = (
        df_sub.groupby(pid_col, as_index=False)[value_col]
        .mean()
        .dropna()
    )
    x = pid_means[value_col].to_numpy(dtype=float)
    n = len(x)
    if n == 0:
        return np.nan, np.nan, np.nan, 0
    point = float(x.mean())
    if n == 1:
        return point, np.nan, np.nan, 1

    rng = np.random.default_rng(seed)
    boots = rng.choice(x, size=(n_boot, n), replace=True).mean(axis=1)
    lo, hi = np.quantile(boots, [0.025, 0.975])
    return point, float(lo), float(hi), int(n)

def summarize_trials(df_in, value_col, n_boot=10000, seed=0):
    rows = []
    for (g, t), sub in df_in.groupby(["group", "task_type"], observed=True):
        cell_seed = seed + 10007*int(g) + (hash(str(t)) % 1_000_000)
        m, lo, hi, n = pid_boot_ci95(sub, value_col, n_boot=n_boot, seed=cell_seed)
        rows.append({"group": int(g), "task_type": str(t), "mean": m, "ci_lo": lo, "ci_hi": hi, "n": n})
    return pd.DataFrame(rows)

# -----------------------------
# Summaries (participant-level bootstrap)
# -----------------------------
N_BOOT = 10000
SEED = 0

acc_group = summarize_trials(df, "is_correct", n_boot=N_BOOT, seed=SEED)
rt_group  = summarize_trials(df, "rt_seconds", n_boot=N_BOOT, seed=SEED + 12345)

# -----------------------------
# ORDER: rank by SAME (Group 1) mean, highest -> lowest
# -----------------------------
acc_order = (
    acc_group[acc_group["group"] == 1]
    .sort_values("mean", ascending=False)["task_type"]
    .tolist()
)

rt_order = (
    rt_group[rt_group["group"] == 1]
    .sort_values("mean", ascending=True)["task_type"]
    .tolist()
)

# -----------------------------
# Plotting function: 2 series only (Same vs Different)
# -----------------------------
def plot_two_series(order, group_df, ylabel, title, out_prefix, *, ylim=None, yticks=None):
    plt.figure(figsize=FIGSIZE)
    x = np.arange(len(order))

    handles = []
    labels  = []

    for g in [1, 2]:
        subg = group_df[group_df["group"] == g].set_index("task_type").reindex(order)

        y  = subg["mean"].to_numpy()
        lo = subg["ci_lo"].to_numpy()
        hi = subg["ci_hi"].to_numpy()
        yerr = np.vstack([y - lo, hi - y])

        h = plt.errorbar(
            x + OFFSETS[g],
            y,
            yerr=yerr,
            fmt="o",
            markersize=MARKERSIZE,
            color=group_colors[g],
            alpha=GROUP_ALPHA if g == 2 else 0.85,
            elinewidth=ELINEWIDTH,
            capsize=CAPSIZE,          # <-- no caps/hats
            zorder=2
        )

        handles.append(h)
        labels.append(group_labels[g])

    plt.xticks(x, order, rotation=20, ha="right", fontsize=TICK_FONTSIZE)
    plt.yticks(fontsize=TICK_FONTSIZE)
    plt.ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    plt.title(title, fontsize=TITLE_FONTSIZE)
    plt.grid(False)

    if ylim is not None:
        plt.ylim(*ylim)
    if yticks is not None:
        plt.yticks(yticks, fontsize=TICK_FONTSIZE)

    plt.legend(handles=handles, labels=labels, frameon=False, fontsize=LEGEND_FONTSIZE, loc="best")

    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{out_prefix}.png", dpi=DPI, bbox_inches="tight")
    plt.savefig(f"{FIG_DIR}/{out_prefix}.pdf", bbox_inches="tight")
    plt.show()

# -----------------------------
# Make plots
# -----------------------------
# Accuracy: y-axis spec + y-lim 0..1 + show ticks
plot_two_series(
    order=acc_order,
    group_df=acc_group,
    ylabel="proportion correct",
    title="",
    out_prefix="fig_accuracy_by_type_pidboot_no_overall",
    ylim=(0, 1),
    yticks=np.linspace(0, 1, 6)  # 0.0,0.2,...,1.0
)

print(f"Saved figures to: {FIG_DIR}/")

## Speech Rate Across Problem-Solving Phases

In [ ]:
# ============================================================
# Config
# ============================================================
PHASE_ORDER = ["pre_incorrect", "first_correct", "post_correct", "post_incorrect", "unlabeled"]
RECORDINGS_PATH = DATA_DIR / "recordings"
USE_AUDIO = True  # if False, wps_used will use rt_seconds instead


# ============================================================
# 1) Add phase labels relative to first correct trial
# ============================================================
def add_first_success_phase(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["participant_id"] = out["participant_id"].astype(str)
    out["trial_number"] = pd.to_numeric(out["trial_number"], errors="coerce").astype("Int64")
    out["is_correct"] = (
        out["is_correct"]
        .map({True: True, False: False, "True": True, "False": False, 1: True, 0: False, "1": True, "0": False})
        .fillna(out["is_correct"])
        .astype(bool)
    )

    first_success = (
        out.loc[out["trial_number"].notna() & out["is_correct"]]
        .groupby("participant_id")["trial_number"]
        .min()
    )
    out["first_success_trial"] = out["participant_id"].map(first_success)

    out["phase"] = "unlabeled"
    idx = out.index[out["trial_number"].notna()]

    t = out.loc[idx, "trial_number"].astype(float)
    fs = pd.to_numeric(out.loc[idx, "first_success_trial"], errors="coerce")
    corr = out.loc[idx, "is_correct"].astype(bool)

    labels = np.select(
        [
            np.asarray(((t < fs) & (~corr)).fillna(False), dtype=bool),
            np.asarray(((t == fs) & corr).fillna(False), dtype=bool),
            np.asarray(((t > fs) & corr).fillna(False), dtype=bool),
            np.asarray(((t > fs) & (~corr)).fillna(False), dtype=bool),
        ],
        ["pre_incorrect", "first_correct", "post_correct", "post_incorrect"],
        default="unlabeled",
    )

    out.loc[idx, "phase"] = labels
    out["phase"] = pd.Categorical(out["phase"], categories=PHASE_ORDER, ordered=True)
    return out


trials_df_speech_all = add_first_success_phase(trials_df.copy())


# ============================================================
# 2) Load transcripts and count words
# ============================================================
def load_full_transcript(pid: str, trial_number: int, recordings_path: Path):
    path = recordings_path / pid / f"trial_{trial_number}_transcription_whisperX.json"
    try:
        with open(path, "r") as f:
            data = json.load(f)
        return data.get("without_timestamps", {}).get("text")
    except Exception:
        return None


def count_words(text):
    return len(re.findall(r"\b\w+\b", text)) if isinstance(text, str) else 0


transcripts = []
for pid, trial_number in tqdm(
    trials_df_speech_all[["participant_id", "trial_number"]].itertuples(index=False),
    total=len(trials_df_speech_all),
):
    if pd.isna(trial_number):
        transcripts.append(None)
    else:
        transcripts.append(load_full_transcript(pid, int(trial_number), RECORDINGS_PATH))

trials_df_speech_all["full_transcript"] = transcripts
trials_df_speech_all["word_count"] = trials_df_speech_all["full_transcript"].apply(count_words)


# ============================================================
# 3) Exclude participants with any zero-word trial
# ============================================================
zero_word_pids = trials_df_speech_all.loc[
    trials_df_speech_all["word_count"].eq(0), "participant_id"
].unique()

trials_df_speech_all = trials_df_speech_all.loc[
    ~trials_df_speech_all["participant_id"].isin(zero_word_pids)
].copy()


# ============================================================
# 4) Add audio duration from speech annotations
# ============================================================
def load_trial_annotation(pid: str, trial_number: int, recordings_path: Path):
    path = recordings_path / pid / f"trial_{trial_number}_speech_annotated.json"
    if not path.exists():
        return None
    try:
        with open(path, "r") as f:
            data = json.load(f)
        data["speech_segments"] = data.get("speech_segments") or []
        data["silence_segments"] = data.get("silence_segments") or []
        return data
    except Exception:
        return None


def infer_audio_duration(speech_segments, silence_segments):
    ends = [
        float(seg["end"])
        for seg in (speech_segments + silence_segments)
        if isinstance(seg.get("end"), (int, float)) and seg["end"] >= 0
    ]
    return max(ends) if ends else np.nan


def build_audio_duration_df(df: pd.DataFrame, recordings_path: Path) -> pd.DataFrame:
    keys = (
        df[["participant_id", "trial_number"]]
        .dropna(subset=["trial_number"])
        .drop_duplicates()
        .copy()
    )
    keys["participant_id"] = keys["participant_id"].astype(str)
    keys["trial_number"] = pd.to_numeric(keys["trial_number"], errors="coerce").astype("Int64")

    rows = []
    for pid, trial_number in keys.itertuples(index=False):
        ann = load_trial_annotation(pid, int(trial_number), recordings_path)
        if ann is None:
            rows.append({"participant_id": pid, "trial_number": int(trial_number), "audio_duration_s": np.nan})
            continue

        rows.append({
            "participant_id": pid,
            "trial_number": int(trial_number),
            "audio_duration_s": infer_audio_duration(
                ann.get("speech_segments", []),
                ann.get("silence_segments", []),
            ),
        })

    return pd.DataFrame(rows)


audio_df = build_audio_duration_df(trials_df_speech_all, RECORDINGS_PATH)

trials_df_speech_all = (
    trials_df_speech_all
    .drop(columns=["audio_duration_s"], errors="ignore")
    .merge(audio_df, on=["participant_id", "trial_number"], how="left", validate="many_to_one")
)


# ============================================================
# 5) Export trial-level speech-rate table for R
# ============================================================
df = trials_df_speech_all.copy()

needed = [
    "participant_id", "trial_number", "phase", "group", "task_type", "is_correct",
    "word_count", "rt_seconds", "audio_duration_s",
]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in trials_df_speech_all: {missing}")

df["participant_id"] = df["participant_id"].astype(str)
df["group"] = pd.to_numeric(df["group"], errors="coerce")
df["is_correct"] = (pd.to_numeric(df["is_correct"], errors="coerce") > 0).astype(int)
df["word_count"] = pd.to_numeric(df["word_count"], errors="coerce")
df["rt_seconds"] = pd.to_numeric(df["rt_seconds"], errors="coerce")
df["audio_duration_s"] = pd.to_numeric(df["audio_duration_s"], errors="coerce")

df["wps_rt"] = df["word_count"] / df["rt_seconds"]
df["wps_audio"] = df["word_count"] / df["audio_duration_s"]
df["wps_used"] = np.where(USE_AUDIO, df["wps_audio"], df["wps_rt"])
df["wps_source"] = "audio" if USE_AUDIO else "rt"

df = df[(df["rt_seconds"] > 0) & (df["audio_duration_s"] > 0) & (df["word_count"] >= 0)].copy()
df["group"] = df["group"].astype(int)

cols = [
    "participant_id", "trial_number", "phase",
    "group", "task_type", "is_correct",
    "word_count", "rt_seconds", "audio_duration_s",
    "wps_rt", "wps_audio", "wps_used", "wps_source",
]

out_path = ANALYSIS_RESULTS_DIR / "speech_rate" / "speech_rate_trial_level.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df[cols].to_csv(out_path, index=False)

## Speech Density Across Problem-Solving Phases

In [ ]:
# ============================================================
# Config
# ============================================================
FIG_DIR = FIGURES_DIR / "speech_density"
DPI = globals().get("DPI", 600)

K = 5
LEXICAL_ONLY = False
N_BOOT = 10_000
SEED = 42

GROUP_ORDER = [1, 2]
group_labels = {1: "Same Type (Group 1)", 2: "Different Types (Group 2)"}

PHASES_FOR_BINS = ["pre_incorrect", "first_correct", "post_correct"]
PHASE_LABELS = {
    "pre_incorrect": "pre-success",
    "first_correct": "first-success",
    "post_correct": "post-success",
}


# ============================================================
# 1) Trial-level speech-density features
# ============================================================
def _sum_duration(segments, lexical=None):
    total = 0.0
    for seg in segments:
        if lexical is not None:
            if "lexical" not in seg or bool(seg["lexical"]) != bool(lexical):
                continue
        dur = seg.get("duration")
        if isinstance(dur, (int, float)) and dur >= 0:
            total += float(dur)
    return total


def build_speech_density_trial_df(df_trials: pd.DataFrame, recordings_path: Path) -> pd.DataFrame:
    required = {"participant_id", "trial_number", "audio_duration_s"}
    missing = required - set(df_trials.columns)
    if missing:
        raise ValueError(f"Missing required columns in trials_df_speech_all: {missing}")

    keys = (
        df_trials[["participant_id", "trial_number"]]
        .copy()
        .assign(
            participant_id=lambda d: d["participant_id"].astype(str),
            trial_number=lambda d: pd.to_numeric(d["trial_number"], errors="coerce").astype("Int64"),
        )
        .dropna(subset=["trial_number"])
        .drop_duplicates()
    )

    rows = []
    for pid, trial_number in keys.itertuples(index=False):
        ann = load_trial_annotation(pid, int(trial_number), recordings_path)
        if ann is None:
            rows.append({
                "participant_id": pid,
                "trial_number": int(trial_number),
                "speech_total_s": np.nan,
                "silence_total_s": np.nan,
                "lexical_total_s": np.nan,
                "nonlex_total_s": np.nan,
            })
            continue

        speech = ann.get("speech_segments", [])
        silence = ann.get("silence_segments", [])

        rows.append({
            "participant_id": pid,
            "trial_number": int(trial_number),
            "speech_total_s": _sum_duration(speech),
            "silence_total_s": _sum_duration(silence),
            "lexical_total_s": _sum_duration(speech, lexical=True),
            "nonlex_total_s": _sum_duration(speech, lexical=False),
        })

    feats = pd.DataFrame(rows)

    out = df_trials.copy().merge(
        feats,
        on=["participant_id", "trial_number"],
        how="left",
        validate="many_to_one",
    )

    out["audio_duration_s"] = pd.to_numeric(out["audio_duration_s"], errors="coerce")
    positive_audio = out["audio_duration_s"] > 0

    out["speech_ratio_audio"] = np.where(
        positive_audio, out["speech_total_s"] / out["audio_duration_s"], np.nan
    )
    out["lexical_ratio_audio"] = np.where(
        positive_audio, out["lexical_total_s"] / out["audio_duration_s"], np.nan
    )
    out["nonlex_ratio_audio"] = np.where(
        positive_audio, out["nonlex_total_s"] / out["audio_duration_s"], np.nan
    )

    return out


trials_speech_silence_df = build_speech_density_trial_df(trials_df_speech_all, RECORDINGS_PATH)


# ============================================================
# 2) K=5 binning over trial progress
# ============================================================
def _overlap_len(seg_start, seg_end, win_start, win_end):
    return max(0.0, min(seg_end, win_end) - max(seg_start, win_start))


def speech_density_bins_from_segments(speech_segments, duration_s, K=5, lexical_only=False):
    if not isinstance(duration_s, (int, float)) or np.isnan(duration_s) or duration_s <= 0:
        return np.full(K, np.nan, dtype=float)

    duration_s = float(duration_s)
    edges = np.linspace(0.0, duration_s, K + 1)
    bin_len = duration_s / K
    speech_time = np.zeros(K, dtype=float)

    for seg in speech_segments:
        s = seg.get("start")
        e = seg.get("end")
        if not (isinstance(s, (int, float)) and isinstance(e, (int, float))):
            continue
        s = float(s)
        e = float(e)
        if e <= s:
            continue

        if lexical_only:
            if "lexical" not in seg or bool(seg["lexical"]) is not True:
                continue

        if e <= 0 or s >= duration_s:
            continue
        s = max(0.0, s)
        e = min(duration_s, e)
        if e <= s:
            continue

        i0 = max(0, min(K - 1, int(np.floor((s / duration_s) * K))))
        i1 = max(0, min(K - 1, int(np.floor((e / duration_s) * K))))

        for i in range(i0, i1 + 1):
            speech_time[i] += _overlap_len(s, e, edges[i], edges[i + 1])

    dens = speech_time / bin_len if bin_len > 0 else np.full(K, np.nan, dtype=float)
    return np.clip(dens, 0.0, 1.0)


def build_bin_density_long_df(df_trials, recordings_path, K=5, lexical_only=False):
    base = df_trials.copy()

    required = ["participant_id", "trial_number", "group", "phase", "audio_duration_s", "task_type"]
    missing = [c for c in required if c not in base.columns]
    if missing:
        raise KeyError(f"Missing required columns in df_trials: {missing}")

    base = base[base["phase"].isin(PHASES_FOR_BINS)].copy()
    base["audio_duration_s"] = pd.to_numeric(base["audio_duration_s"], errors="coerce")
    base = base[base["audio_duration_s"].notna() & (base["audio_duration_s"] > 0)].copy()

    base["participant_id"] = base["participant_id"].astype(str)
    base["trial_number"] = pd.to_numeric(base["trial_number"], errors="coerce")
    base["group"] = pd.to_numeric(base["group"], errors="coerce")
    base = base.dropna(subset=["trial_number", "group"]).copy()
    base["trial_number"] = base["trial_number"].astype(int)
    base["group"] = base["group"].astype(int)
    base["task_type"] = base["task_type"].astype(str)

    rows = []
    for pid, trial_number, group, phase, duration_s, task_type in base[
        ["participant_id", "trial_number", "group", "phase", "audio_duration_s", "task_type"]
    ].itertuples(index=False):
        ann = load_trial_annotation(pid, int(trial_number), recordings_path)
        if ann is None:
            continue

        dens = speech_density_bins_from_segments(
            ann.get("speech_segments", []),
            float(duration_s),
            K=K,
            lexical_only=lexical_only,
        )
        if not np.all(np.isfinite(dens)):
            continue

        for bin_idx, value in enumerate(dens, start=1):
            rows.append({
                "participant_id": pid,
                "trial_number": int(trial_number),
                "group": int(group),
                "phase": str(phase),
                "task_type": str(task_type),
                "bin_idx": int(bin_idx),
                "speech_density": float(value),
            })

    return pd.DataFrame(rows)


long_bins = build_bin_density_long_df(
    trials_speech_silence_df,
    RECORDINGS_PATH,
    K=K,
    lexical_only=LEXICAL_ONLY,
)


# ============================================================
# 3) Participant-bootstrap summary for plotting
# ============================================================
def _stable_seed(*parts, base_seed=0):
    s = "|".join(map(str, parts)) + f"|{base_seed}"
    return int(hashlib.md5(s.encode("utf-8")).hexdigest()[:8], 16)


def bootstrap_mean_ci95(x, n_boot=10000, seed=0):
    x = pd.to_numeric(pd.Series(x), errors="coerce").dropna().to_numpy(dtype=float)
    n = len(x)
    if n == 0:
        return np.nan, np.nan, np.nan, 0
    m = float(x.mean())
    if n == 1:
        return m, np.nan, np.nan, 1

    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = x[idx].mean(axis=1)
    lo, hi = np.percentile(boot_means, [2.5, 97.5])
    return m, float(lo), float(hi), int(n)


def summarize_bins_participant_bootstrap(long_df, n_boot=10000, seed=0):
    per_pid = (
        long_df.groupby(["participant_id", "phase", "group", "bin_idx"], observed=True)["speech_density"]
        .mean()
        .reset_index()
    )

    rows = []
    for (phase, group, bin_idx), sub in per_pid.groupby(["phase", "group", "bin_idx"], observed=True):
        m, lo, hi, n = bootstrap_mean_ci95(
            sub["speech_density"],
            n_boot=n_boot,
            seed=_stable_seed(phase, int(group), int(bin_idx), base_seed=seed),
        )
        rows.append({
            "phase": str(phase),
            "group": int(group),
            "bin_idx": int(bin_idx),
            "mean": m,
            "lo": lo,
            "hi": hi,
            "n_participants": n,
        })

    summary = pd.DataFrame(rows)
    summary["phase"] = pd.Categorical(summary["phase"], categories=PHASES_FOR_BINS, ordered=True)
    return summary.sort_values(["phase", "group", "bin_idx"])


summary_bins = summarize_bins_participant_bootstrap(long_bins, n_boot=N_BOOT, seed=SEED)

In [ ]:
# ============================================================
# 4) Final plot only
# ============================================================
FIGSIZE_GROUPPAN = (7.8, 3.6)
TITLE_FONTSIZE = 18
LABEL_FONTSIZE = 16
TICK_FONTSIZE = 14
LEGEND_FONTSIZE = 13
LINEWIDTH = 2.2
MARKERSIZE = 5.5
BIN_SPACING = 0.72


def compressed_x(K, spacing=BIN_SPACING):
    return 1.0 + np.arange(K) * float(spacing)


def compressed_xlim(K, spacing=BIN_SPACING, pad=0.35):
    x = compressed_x(K, spacing)
    return (x.min() - pad, x.max() + pad)


def plot_group_panels_1x2_three_phases(
    summary_df,
    K,
    lexical_only,
    phases=("pre_incorrect", "first_correct", "post_correct"),
    *,
    out_prefix="speech_density_group_panels",
    ylim=(0.0, 0.6),
    ytick_step=0.1,
):
    def rgb255(r, g, b):
        return (r / 255.0, g / 255.0, b / 255.0)

    phase_linestyles = {
        "pre_incorrect": "-",
        "first_correct": "-",
        "post_correct": "-",
    }

    red_shades = {
        "pre_incorrect": rgb255(163, 44, 44),
        "first_correct": rgb255(240, 105, 14),
        "post_correct": rgb255(240, 12, 12),
    }
    blue_shades = {
        "pre_incorrect": rgb255(50, 20, 110),
        "first_correct": rgb255(34, 160, 245),
        "post_correct": rgb255(29, 67, 191),
    }

    group_phase_colors = {1: red_shades, 2: blue_shades}
    phase_alpha_line = {"pre_incorrect": 0.55, "first_correct": 1.00, "post_correct": 1.00}
    phase_alpha_band = {"pre_incorrect": 0.04, "first_correct": 0.08, "post_correct": 0.08}

    x = compressed_x(K)
    xlims = compressed_xlim(K)

    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_GROUPPAN, sharex=True, sharey=True)

    for ax, grp in zip(axes, GROUP_ORDER):
        ax.set_title(group_labels.get(grp, f"Group {grp}"), fontsize=TITLE_FONTSIZE, pad=8)

        for ph in phases:
            sub = (
                summary_df[(summary_df["group"] == grp) & (summary_df["phase"] == ph)]
                .sort_values("bin_idx")
            )

            y = np.full(K, np.nan, dtype=float)
            lo_ = np.full(K, np.nan, dtype=float)
            hi_ = np.full(K, np.nan, dtype=float)

            for _, r in sub.iterrows():
                bi = int(r["bin_idx"]) - 1
                if 0 <= bi < K:
                    y[bi] = float(r["mean"])
                    lo_[bi] = float(r["lo"])
                    hi_[bi] = float(r["hi"])

            ok_mean = np.isfinite(y)
            ok_band = ok_mean & np.isfinite(lo_) & np.isfinite(hi_)

            color = group_phase_colors.get(int(grp), {}).get(ph, "black")

            if ok_mean.any():
                ax.plot(
                    x[ok_mean],
                    y[ok_mean],
                    marker="o",
                    linewidth=LINEWIDTH,
                    markersize=MARKERSIZE,
                    linestyle=phase_linestyles.get(ph, "-"),
                    color=color,
                    alpha=phase_alpha_line.get(ph, 1.0),
                )

            if ok_band.any():
                ax.fill_between(
                    x[ok_band],
                    lo_[ok_band],
                    hi_[ok_band],
                    color=color,
                    alpha=phase_alpha_band.get(ph, 0.08),
                    linewidth=0,
                )

        ax.set_xlim(*xlims)
        ax.set_ylim(*ylim)
        ax.grid(False)
        ax.set_xticks(x)
        ax.set_xticklabels([str(i) for i in range(1, K + 1)], fontsize=TICK_FONTSIZE)

        if ytick_step is not None and ytick_step > 0:
            ax.set_yticks(np.arange(ylim[0], ylim[1] + 1e-9, ytick_step))
        ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)
        ax.set_xlabel(f"trial progress (bin 1..{K})", fontsize=LABEL_FONTSIZE)

    axes[0].set_ylabel("speech density", fontsize=LABEL_FONTSIZE)

    handles, names = [], []
    for ph in phases:
        handles.append(
            Line2D(
                [0], [0],
                color=red_shades.get(ph, "black"),
                linestyle=phase_linestyles.get(ph, "-"),
                marker="o",
                linewidth=LINEWIDTH,
                markersize=MARKERSIZE,
                alpha=phase_alpha_line.get(ph, 1.0),
            )
        )
        names.append(PHASE_LABELS.get(ph, ph))

    fig.legend(
        handles,
        names,
        loc="upper center",
        ncol=len(phases),
        frameon=False,
        bbox_to_anchor=(0.5, 0.93),
        fontsize=LEGEND_FONTSIZE,
        handlelength=2.8,
        columnspacing=1.8,
    )

    plt.tight_layout(rect=[0, 0, 1, 0.88])

    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / f"{out_prefix}.png", dpi=DPI, bbox_inches="tight", facecolor="white")
    fig.savefig(FIG_DIR / f"{out_prefix}.pdf", bbox_inches="tight", facecolor="white")

    plt.show()


plot_group_panels_1x2_three_phases(
    summary_bins,
    K=K,
    lexical_only=LEXICAL_ONLY,
    phases=("pre_incorrect", "first_correct", "post_correct"),
    out_prefix="speech_density_group_panels",
)

In [ ]:
# ============================================================
# 5) Export trial-level and bin-level files
# ============================================================
trial_cols = [
    "participant_id", "group", "trial_number", "task_type",
    "phase", "is_correct",
    "audio_duration_s", "speech_total_s", "silence_total_s",
    "speech_ratio_audio",
    "lexical_total_s", "nonlex_total_s",
    "lexical_ratio_audio", "nonlex_ratio_audio",
]

trial_out = trials_speech_silence_df[[c for c in trial_cols if c in trials_speech_silence_df.columns]].copy()
trial_out["is_correct"] = trial_out["is_correct"].astype(int)

trial_path = ANALYSIS_RESULTS_DIR / "speech_density" / "speech_density_trial_level.csv"
trial_path.parent.mkdir(parents=True, exist_ok=True)
trial_out.to_csv(trial_path, index=False)

bin_cols = [
    "participant_id", "group", "trial_number", "task_type",
    "phase", "bin_idx", "speech_density",
]

bin_out = long_bins[[c for c in bin_cols if c in long_bins.columns]].copy()

bin_path = ANALYSIS_RESULTS_DIR / "speech_density" / f"speech_density_bin_level_K{K}.csv"
bin_path.parent.mkdir(parents=True, exist_ok=True)
bin_out.to_csv(bin_path, index=False)
print(f"Speech-density data exported to {trial_path.parent}")

## Semantic Classifier Analyses

In [ ]:
# ============================================================
# Semantic Analysis: Load Cleaned Cached Embeddings
# ============================================================
EMBED_DIR = SPEECH_SEMANTICS_DIR / "embeddings" / "utterance_level" / "openai_text_embedding_3_large_cleaned"
META_PATH = EMBED_DIR / "utterance_metadata.parquet"
EMB_PATH = EMBED_DIR / "utterance_embeddings.npy"
MANIFEST_PATH = EMBED_DIR / "manifest.json"

with open(MANIFEST_PATH, "r") as f:
    embed_manifest = json.load(f)

utterances_meta = pd.read_parquet(META_PATH)
utterance_emb = np.load(EMB_PATH)

assert len(utterances_meta) == utterance_emb.shape[0]
assert "problem_key" in utterances_meta.columns
assert "input" in utterances_meta.columns

print(
    f"Loaded cleaned cached embeddings: "
    f"{len(utterances_meta)} utterances across "
    f"{utterances_meta['participant_id'].astype(str).nunique()} participants."
)


In [ ]:
# ============================================================
# Shared helpers
# ============================================================
PID_COL = "participant_id"
GROUP_COL = "group"
TRIAL_COL = "trial_number"
TASK_COL = "task_type"
PHASE_COL = "phase"
CORRECT_COL = "is_correct"

SAME_GROUP_VALUE = 1
DIFF_GROUP_VALUE = 2

PHASE_PRE = "pre_incorrect"
PHASE_FIRST = "first_correct"
PHASE_POST = "post_correct"

N_SPLITS = 5
C = 1.0
MAX_ITER = 5000
SOLVER = "lbfgs"
RNG_SEED = 42


def _stable_seed(*parts, base=0):
    s = "|".join(map(str, parts)) + f"|{base}"
    return int(hashlib.md5(s.encode("utf-8")).hexdigest()[:8], 16)


def _build_cov_preprocessor(df: pd.DataFrame, *, include_task_type: bool, include_trial_number: bool):
    cat_cols = []
    num_cols = []

    if include_task_type and (TASK_COL in df.columns):
        cat_cols.append(TASK_COL)
    if include_trial_number and (TRIAL_COL in df.columns):
        num_cols.append(TRIAL_COL)

    cov_pre = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
            ("num", StandardScaler(), num_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )
    return cov_pre, cat_cols, num_cols


def _weights_participant_equal(df_train: pd.DataFrame):
    counts = df_train[PID_COL].astype(str).value_counts()
    return df_train[PID_COL].astype(str).map(lambda pid: 1.0 / counts[pid]).to_numpy(dtype=float)


def _weights_participant_class_equal(df_train: pd.DataFrame, y_train: np.ndarray):
    pid = df_train[PID_COL].astype(str).to_numpy()
    y = np.asarray(y_train).astype(int)
    key = pd.Series(pid).astype(str) + "||" + pd.Series(y).astype(str)
    counts = key.value_counts()
    return key.map(lambda k: 1.0 / counts[k]).to_numpy(dtype=float)


def participant_bootstrap_ci_fixed_preds(
    df_sub: pd.DataFrame,
    y: np.ndarray,
    p: np.ndarray,
    *,
    n_boot: int,
    seed: int,
):
    rng = np.random.default_rng(seed)

    pids = df_sub[PID_COL].astype(str).to_numpy()
    uniq = np.unique(pids)
    pid_to_idx = {pid: np.flatnonzero(pids == pid) for pid in uniq}

    boot_acc = np.empty(n_boot, dtype=float)
    boot_auc = np.full(n_boot, np.nan, dtype=float)

    for b in range(n_boot):
        samp = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([pid_to_idx[sp] for sp in samp])

        yy = y[idx]
        pp = p[idx]

        boot_acc[b] = accuracy_score(yy, (pp >= 0.5).astype(int))
        if len(np.unique(yy)) > 1:
            boot_auc[b] = roc_auc_score(yy, pp)

    acc_point = float(accuracy_score(y, (p >= 0.5).astype(int)))
    auc_point = float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan

    acc_lo, acc_hi = np.quantile(boot_acc, [0.025, 0.975])

    auc_vals = boot_auc[np.isfinite(boot_auc)]
    if auc_vals.size > 0:
        auc_lo, auc_hi = np.quantile(auc_vals, [0.025, 0.975])
    else:
        auc_lo, auc_hi = (np.nan, np.nan)

    return {
        "acc": acc_point,
        "acc_ci_lo": float(acc_lo),
        "acc_ci_hi": float(acc_hi),
        "auc": auc_point,
        "auc_ci_lo": float(auc_lo),
        "auc_ci_hi": float(auc_hi),
    }


def oof_predict_logreg(
    df_sub: pd.DataFrame,
    X_sub: np.ndarray,
    *,
    include_task_type: bool,
    include_trial_number: bool,
    weight_mode: str,
    fold_assignments: pd.DataFrame | None = None,
    analysis_name: str | None = None,
):
    df_sub = df_sub.reset_index(drop=False).rename(columns={"index": "row_index"}).copy()
    y = df_sub["y"].astype(int).to_numpy()

    cov_pre, cat_cols, num_cols = _build_cov_preprocessor(
        df_sub,
        include_task_type=include_task_type,
        include_trial_number=include_trial_number,
    )

    p_oof = np.full(len(df_sub), np.nan, dtype=float)

    if fold_assignments is not None:
        if analysis_name is None:
            raise ValueError("analysis_name is required when using fixed fold assignments.")

        fa = fold_assignments.loc[
            fold_assignments["analysis"] == analysis_name,
            ["row_index", "fold"],
        ].copy()
        df_sub = df_sub.merge(fa, on="row_index", how="left", validate="one_to_one")

        if df_sub["fold"].isna().any():
            missing = int(df_sub["fold"].isna().sum())
            raise ValueError(f"Missing fixed fold assignment for {missing} rows in {analysis_name}.")

        df_sub["fold"] = df_sub["fold"].astype(int)
        splits = []
        for fold in sorted(df_sub["fold"].unique()):
            te_idx = df_sub.index[df_sub["fold"] == fold].to_numpy()
            tr_idx = df_sub.index[df_sub["fold"] != fold].to_numpy()
            splits.append((tr_idx, te_idx))
    else:
        groups = df_sub[PID_COL].astype(str).to_numpy()
        gkf = GroupKFold(n_splits=N_SPLITS)
        splits = list(gkf.split(df_sub, y, groups=groups))

    for tr_idx, te_idx in splits:
        df_tr = df_sub.iloc[tr_idx].copy()
        df_te = df_sub.iloc[te_idx].copy()

        if len(cat_cols) + len(num_cols) > 0:
            Xcov_tr = cov_pre.fit_transform(df_tr)
            Xcov_te = cov_pre.transform(df_te)
            Xcov_tr = Xcov_tr.toarray() if hasattr(Xcov_tr, "toarray") else np.asarray(Xcov_tr)
            Xcov_te = Xcov_te.toarray() if hasattr(Xcov_te, "toarray") else np.asarray(Xcov_te)
        else:
            Xcov_tr = np.zeros((len(tr_idx), 0), dtype=float)
            Xcov_te = np.zeros((len(te_idx), 0), dtype=float)

        emb_scaler = StandardScaler()
        Xemb_tr = emb_scaler.fit_transform(X_sub[tr_idx])
        Xemb_te = emb_scaler.transform(X_sub[te_idx])

        Xtr = np.hstack([Xcov_tr, Xemb_tr])
        Xte = np.hstack([Xcov_te, Xemb_te])

        if weight_mode == "participant_equal":
            w_tr = _weights_participant_equal(df_tr)
        elif weight_mode == "participant_class_equal":
            w_tr = _weights_participant_class_equal(df_tr, y[tr_idx])
        else:
            raise ValueError(f"Unknown weight_mode: {weight_mode}")

        clf = LogisticRegression(
            penalty="l2",
            C=C,
            solver=SOLVER,
            max_iter=MAX_ITER,
            class_weight=None,
        )
        clf.fit(Xtr, y[tr_idx], sample_weight=w_tr)
        p_oof[te_idx] = clf.predict_proba(Xte)[:, 1]

    assert np.isfinite(p_oof).all()
    return p_oof


def _problem_level_bootstrap_ci(problem_df: pd.DataFrame, metric_col: str, *, n_boot: int, seed: int):
    rng = np.random.default_rng(seed)
    vals = problem_df[metric_col].to_numpy(dtype=float)
    n = len(vals)
    if n == 0:
        raise ValueError("No problems available for bootstrap.")

    boot = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        samp = rng.choice(vals, size=n, replace=True)
        boot[b] = samp.mean()

    point = float(vals.mean())
    ci_lo, ci_hi = np.quantile(boot, [0.025, 0.975])
    return point, float(ci_lo), float(ci_hi), boot

In [ ]:
# ============================================================
# Same vs Different on the same problem (Trials 2-5)
# ============================================================
PROBLEM_COL = "problem_key"
TRIAL_MIN = 2
INCLUDE_TASK_TYPE_PROBLEM = False
INCLUDE_TRIAL_NUMBER_PROBLEM = True
N_BOOT_PROBLEMS = 1000

SAVE_DIR_PROBLEM = SPEECH_SEMANTICS_RESULTS_DIR / "same_vs_diff_by_problem"
SAVE_TAG_PROBLEM = "problem_level_bootstrap_equal_pid_weight_with_per_problem_cis"
SAVE_DIR_PROBLEM.mkdir(parents=True, exist_ok=True)


def make_problem_df(
    utterances_meta: pd.DataFrame,
    *,
    problem_value,
    problem_col: str,
    trial_min: int,
    same_group_value: int,
    diff_group_value: int,
):
    df = utterances_meta.copy()

    need_cols = [PID_COL, GROUP_COL, TRIAL_COL, problem_col]
    for c in need_cols:
        if c not in df.columns:
            raise ValueError(f"utterances_meta missing required column: {c}")

    df = df[df[problem_col] == problem_value].copy()
    df[TRIAL_COL] = pd.to_numeric(df[TRIAL_COL], errors="coerce")
    df = df[df[TRIAL_COL] >= trial_min].copy()
    df = df[df[GROUP_COL].isin([same_group_value, diff_group_value])].copy()
    df["y"] = (df[GROUP_COL] == same_group_value).astype(int)

    if df["y"].nunique() < 2:
        return None

    pid_class = df.groupby(PID_COL)["y"].nunique()
    bad_pids = pid_class[pid_class > 1].index.astype(str).tolist()
    if bad_pids:
        raise ValueError(
            f"Some participants appear in both SAME and DIFF within problem={problem_value}. "
            f"Example bad pids: {bad_pids[:5]}"
        )

    return df


def run_same_vs_diff_problem_pipeline(
    utterances_meta: pd.DataFrame,
    utterance_emb: np.ndarray,
    *,
    problem_col: str = PROBLEM_COL,
    trial_min: int = TRIAL_MIN,
    include_task_type: bool = INCLUDE_TASK_TYPE_PROBLEM,
    include_trial_number: bool = INCLUDE_TRIAL_NUMBER_PROBLEM,
    seed: int = RNG_SEED,
    n_boot_problems: int = N_BOOT_PROBLEMS,
):
    problems = (
        utterances_meta.loc[
            pd.to_numeric(utterances_meta[TRIAL_COL], errors="coerce") >= trial_min,
            problem_col,
        ]
        .dropna()
        .unique()
        .tolist()
    )
    problems = sorted(problems)

    rows = []
    for prob in problems:
        df_sub = make_problem_df(
            utterances_meta,
            problem_value=prob,
            problem_col=problem_col,
            trial_min=trial_min,
            same_group_value=SAME_GROUP_VALUE,
            diff_group_value=DIFF_GROUP_VALUE,
        )
        if df_sub is None or len(df_sub) == 0:
            continue

        n_pids = df_sub[PID_COL].nunique()
        if n_pids < N_SPLITS:
            continue

        keep_idx = df_sub.index.to_numpy()
        X_sub = utterance_emb[keep_idx].astype(np.float32, copy=False)

        p = oof_predict_logreg(
            df_sub.reset_index(drop=True),
            X_sub,
            include_task_type=include_task_type,
            include_trial_number=include_trial_number,
            weight_mode="participant_equal",
        )
        y = df_sub["y"].astype(int).to_numpy()

        ci = participant_bootstrap_ci_fixed_preds(
            df_sub.reset_index(drop=True),
            y,
            p,
            n_boot=n_boot_problems,
            seed=_stable_seed("same_vs_diff_problem", prob, base=seed),
        )

        n_same_pids = int(df_sub.loc[df_sub["y"] == 1, PID_COL].nunique())
        n_diff_pids = int(df_sub.loc[df_sub["y"] == 0, PID_COL].nunique())
        n_same_utts = int((df_sub["y"] == 1).sum())
        n_diff_utts = int((df_sub["y"] == 0).sum())

        assert df_sub[TASK_COL].nunique() == 1, f"problem_key {prob} has multiple task_type values"
        task_type = df_sub[TASK_COL].iloc[0]

        rows.append({
            "problem_key": prob,
            "task_type": task_type,
            "n_participants": int(n_pids),
            "n_same_participants": n_same_pids,
            "n_diff_participants": n_diff_pids,
            "n_utterances": int(len(df_sub)),
            "n_same_utterances": n_same_utts,
            "n_diff_utterances": n_diff_utts,
            "acc": ci["acc"],
            "acc_ci_lo": ci["acc_ci_lo"],
            "acc_ci_hi": ci["acc_ci_hi"],
            "auc": ci["auc"],
            "auc_ci_lo": ci["auc_ci_lo"],
            "auc_ci_hi": ci["auc_ci_hi"],
            "include_trial_number": include_trial_number,
            "include_task_type": include_task_type,
        })

    problem_df = pd.DataFrame(rows).sort_values("problem_key").reset_index(drop=True)

    acc_point, acc_lo, acc_hi, acc_boot = _problem_level_bootstrap_ci(
        problem_df, "acc", n_boot=n_boot_problems, seed=seed
    )
    auc_point, auc_lo, auc_hi, auc_boot = _problem_level_bootstrap_ci(
        problem_df.dropna(subset=["auc"]),
        "auc",
        n_boot=n_boot_problems,
        seed=seed + 1,
    )

    summary_df = pd.DataFrame([
        {
            "metric": "acc_mean_over_problems",
            "n_problems": int(len(problem_df)),
            "mean": acc_point,
            "ci_lo": acc_lo,
            "ci_hi": acc_hi,
            "trial_min": trial_min,
            "include_trial_number": include_trial_number,
            "include_task_type": include_task_type,
            "weighting": "participant_equal (train-fold)",
            "bootstrap_unit": "problem_key",
            "n_boot": n_boot_problems,
        },
        {
            "metric": "auc_mean_over_problems",
            "n_problems": int(problem_df["auc"].notna().sum()),
            "mean": auc_point,
            "ci_lo": auc_lo,
            "ci_hi": auc_hi,
            "trial_min": trial_min,
            "include_trial_number": include_trial_number,
            "include_task_type": include_task_type,
            "weighting": "participant_equal (train-fold)",
            "bootstrap_unit": "problem_key",
            "n_boot": n_boot_problems,
        },
    ])

    n_total = int(len(problem_df))
    n_point_above = int((problem_df["acc"] > 0.5).sum())
    n_ci_above = int((problem_df["acc_ci_lo"] > 0.5).sum())

    sig_df = pd.DataFrame([{
        "threshold": 0.5,
        "n_problems": n_total,
        "n_point_acc_above_threshold": n_point_above,
        "n_ci_low_above_threshold": n_ci_above,
        "prop_point_acc_above_threshold": n_point_above / n_total if n_total > 0 else np.nan,
        "prop_ci_low_above_threshold": n_ci_above / n_total if n_total > 0 else np.nan,
        "ci_type": "participant_bootstrap_within_problem (fixed OOF preds)",
        "n_boot_within_problem": n_boot_problems,
    }])

    boot = {
        "acc_boot_over_problems": acc_boot,
        "auc_boot_over_problems": auc_boot,
    }

    return problem_df, summary_df, sig_df, boot


problem_df, problem_summary_df, sig_df, problem_boot = run_same_vs_diff_problem_pipeline(
    utterances_meta=utterances_meta,
    utterance_emb=utterance_emb,
)

stamp_problem = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
base_problem = f"{SAVE_TAG_PROBLEM}__trials{TRIAL_MIN}plus__{stamp_problem}"

summary_path = SAVE_DIR_PROBLEM / f"{base_problem}__summary.csv"
sig_path = SAVE_DIR_PROBLEM / f"{base_problem}__sig_counts.csv"
problem_path = SAVE_DIR_PROBLEM / f"{base_problem}__per_problem.csv"
boot_path = SAVE_DIR_PROBLEM / f"{base_problem}__boot.npz"
manifest_path = SAVE_DIR_PROBLEM / f"{base_problem}__manifest.json"

problem_summary_df.to_csv(summary_path, index=False)
sig_df.to_csv(sig_path, index=False)
problem_df.to_csv(problem_path, index=False)

np.savez_compressed(
    boot_path,
    acc_boot_over_problems=problem_boot["acc_boot_over_problems"],
    auc_boot_over_problems=problem_boot["auc_boot_over_problems"],
)

problem_manifest = {
    "created_at": stamp_problem,
    "problem_col": PROBLEM_COL,
    "trial_min": TRIAL_MIN,
    "groups": {"same": SAME_GROUP_VALUE, "diff": DIFF_GROUP_VALUE},
    "cv": {"n_splits": N_SPLITS, "group": PID_COL},
    "model": {
        "type": "logistic_regression",
        "penalty": "l2",
        "C": C,
        "solver": SOLVER,
        "max_iter": MAX_ITER,
    },
    "features": {
        "embeddings_model": embed_manifest.get("model"),
        "include_trial_number": INCLUDE_TRIAL_NUMBER_PROBLEM,
        "include_task_type": INCLUDE_TASK_TYPE_PROBLEM,
    },
    "weighting": "participant_equal (computed within each training fold)",
    "bootstrap": {
        "mean_over_problems": {"unit": "problem_key", "n_boot": N_BOOT_PROBLEMS},
        "within_problem_cis": {
            "unit": "participant_id",
            "n_boot": N_BOOT_PROBLEMS,
            "uses_fixed_oof_predictions": True,
        },
    },
    "outputs": {
        "summary_csv": str(summary_path),
        "sig_counts_csv": str(sig_path),
        "per_problem_csv": str(problem_path),
        "boot_npz": str(boot_path),
    },
}

with open(manifest_path, "w") as f:
    json.dump(problem_manifest, f, indent=2)

display(problem_summary_df)
display(sig_df)


In [ ]:
# ============================================================
# Correctness + phase analyses (the semantic results used in the paper figure)
# ============================================================
INCLUDE_TASK_TYPE = True
INCLUDE_TRIAL_NUMBER_FOR_CORRECTNESS = True
INCLUDE_TRIAL_NUMBER_FOR_PHASE = False
N_BOOT = 10000

SAVE_DIR_CP = SPEECH_SEMANTICS_RESULTS_DIR / "correctness_and_phase"
SAVE_DIR_CP.mkdir(parents=True, exist_ok=True)

FIXED_FOLDS_PATH = SPEECH_SEMANTICS_DIR / "results/correctness_phase_logreg_folds.csv"
fixed_folds_df = pd.read_csv(FIXED_FOLDS_PATH)

def make_df_correctness(utterances_meta: pd.DataFrame, *, group_value: int | None):
    df = utterances_meta.copy()
    needed = [PID_COL, GROUP_COL, TRIAL_COL, TASK_COL, CORRECT_COL]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"utterances_meta missing required column: {c}")

    df = df.dropna(subset=needed).copy()

    if group_value is not None:
        df = df[df[GROUP_COL] == group_value].copy()

    df["y"] = df[CORRECT_COL].astype(bool).astype(int)

    if df["y"].nunique() < 2:
        return None
    if df[PID_COL].nunique() < N_SPLITS:
        return None

    return df


def make_df_phase_contrast(
    utterances_meta: pd.DataFrame,
    *,
    phase_a: str,
    phase_b: str,
    group_value: int | None,
):
    df = utterances_meta.copy()
    needed = [PID_COL, GROUP_COL, TRIAL_COL, TASK_COL, PHASE_COL]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"utterances_meta missing required column: {c}")

    df = df.dropna(subset=needed).copy()

    if group_value is not None:
        df = df[df[GROUP_COL] == group_value].copy()

    df = df[df[PHASE_COL].isin([phase_a, phase_b])].copy()
    df["y"] = (df[PHASE_COL] == phase_b).astype(int)

    counts = df.groupby([PID_COL, "y"]).size().unstack(fill_value=0)
    ok_pids = counts[(counts.get(0, 0) > 0) & (counts.get(1, 0) > 0)].index.astype(str)
    df = df[df[PID_COL].astype(str).isin(ok_pids)].copy()

    if len(df) == 0 or df["y"].nunique() < 2:
        return None
    if df[PID_COL].nunique() < N_SPLITS:
        return None

    return df


def run_one_analysis(
    utterances_meta: pd.DataFrame,
    utterance_emb: np.ndarray,
    *,
    analysis_name: str,
    df_sub: pd.DataFrame,
    include_task_type: bool,
    include_trial_number: bool,
    weight_mode: str,
    seed: int,
    n_boot: int,
):
    keep_idx = df_sub.index.to_numpy()
    X_sub = utterance_emb[keep_idx].astype(np.float32, copy=False)

    p = oof_predict_logreg(
        df_sub,
        X_sub,
        include_task_type=include_task_type,
        include_trial_number=include_trial_number,
        weight_mode=weight_mode,
        fold_assignments=fixed_folds_df,
        analysis_name=analysis_name,
    )
    y = df_sub["y"].astype(int).to_numpy()

    ci = participant_bootstrap_ci_fixed_preds(
        df_sub.reset_index(drop=True),
        y,
        p,
        n_boot=n_boot,
        seed=seed,
    )

    return {
        "analysis": analysis_name,
        "n_utterances": int(len(df_sub)),
        "n_participants": int(df_sub[PID_COL].nunique()),
        "include_task_type": bool(include_task_type),
        "include_trial_number": bool(include_trial_number),
        "weight_mode": weight_mode,
        "acc": ci["acc"],
        "acc_ci_lo": ci["acc_ci_lo"],
        "acc_ci_hi": ci["acc_ci_hi"],
        "auc": ci["auc"],
        "auc_ci_lo": ci["auc_ci_lo"],
        "auc_ci_hi": ci["auc_ci_hi"],
    }


results = []

# A) Correct vs incorrect, pooled
dfA = make_df_correctness(utterances_meta, group_value=None)
if dfA is not None:
    results.append(
        run_one_analysis(
            utterances_meta,
            utterance_emb,
            analysis_name="A_correct_vs_incorrect__pooled",
            df_sub=dfA,
            include_task_type=INCLUDE_TASK_TYPE,
            include_trial_number=INCLUDE_TRIAL_NUMBER_FOR_CORRECTNESS,
            weight_mode="participant_equal",
            seed=_stable_seed("A_correct_vs_incorrect__pooled", base=RNG_SEED),
            n_boot=N_BOOT,
        )
    )

# B) Correct vs incorrect, per group
for g, gname in [(SAME_GROUP_VALUE, "same"), (DIFF_GROUP_VALUE, "diff")]:
    dfB = make_df_correctness(utterances_meta, group_value=g)
    if dfB is not None:
        results.append(
            run_one_analysis(
                utterances_meta,
                utterance_emb,
                analysis_name=f"B_correct_vs_incorrect__group_{gname}",
                df_sub=dfB,
                include_task_type=INCLUDE_TASK_TYPE,
                include_trial_number=INCLUDE_TRIAL_NUMBER_FOR_CORRECTNESS,
                weight_mode="participant_equal",
                seed=_stable_seed("B_correct_vs_incorrect", gname, base=RNG_SEED),
                n_boot=N_BOOT,
            )
        )

# D) Phase contrasts, pooled + per group
phase_contrasts = [
    (PHASE_PRE, PHASE_FIRST, "D_pre_incorrect_vs_first_correct"),
    (PHASE_FIRST, PHASE_POST, "D_first_correct_vs_post_correct"),
    (PHASE_PRE, PHASE_POST, "D_pre_incorrect_vs_post_correct"),
]

for ph_a, ph_b, label in phase_contrasts:
    dfD = make_df_phase_contrast(utterances_meta, phase_a=ph_a, phase_b=ph_b, group_value=None)
    if dfD is not None:
        results.append(
            run_one_analysis(
                utterances_meta,
                utterance_emb,
                analysis_name=f"{label}__pooled",
                df_sub=dfD,
                include_task_type=INCLUDE_TASK_TYPE,
                include_trial_number=INCLUDE_TRIAL_NUMBER_FOR_PHASE,
                weight_mode="participant_class_equal",
                seed=_stable_seed(label, "pooled", base=RNG_SEED),
                n_boot=N_BOOT,
            )
        )

    for g, gname in [(SAME_GROUP_VALUE, "same"), (DIFF_GROUP_VALUE, "diff")]:
        dfDg = make_df_phase_contrast(utterances_meta, phase_a=ph_a, phase_b=ph_b, group_value=g)
        if dfDg is not None:
            results.append(
                run_one_analysis(
                    utterances_meta,
                    utterance_emb,
                    analysis_name=f"{label}__group_{gname}",
                    df_sub=dfDg,
                    include_task_type=INCLUDE_TASK_TYPE,
                    include_trial_number=INCLUDE_TRIAL_NUMBER_FOR_PHASE,
                    weight_mode="participant_class_equal",
                    seed=_stable_seed(label, gname, base=RNG_SEED),
                    n_boot=N_BOOT,
                )
            )

results_df = pd.DataFrame(results).sort_values("analysis").reset_index(drop=True)

stamp_cp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
base_cp = f"correctness_phase_logreg__{stamp_cp}"

results_path = SAVE_DIR_CP / f"{base_cp}__results.csv"
manifest_path = SAVE_DIR_CP / f"{base_cp}__manifest.json"

results_df.to_csv(results_path, index=False)

cp_manifest = {
    "created_at": stamp_cp,
    "save_dir": str(SAVE_DIR_CP),
    "outputs": {"results_csv": str(results_path)},
    "cv": {"n_splits": N_SPLITS, "group": PID_COL},
    "model": {
        "type": "logistic_regression",
        "penalty": "l2",
        "C": C,
        "solver": SOLVER,
        "max_iter": MAX_ITER,
    },
    "embeddings": {
        "model": embed_manifest.get("model"),
        "dim": int(utterance_emb.shape[1]),
    },
    "covariates": {
        "include_task_type": INCLUDE_TASK_TYPE,
        "include_trial_number_for_correctness": INCLUDE_TRIAL_NUMBER_FOR_CORRECTNESS,
        "include_trial_number_for_phase": INCLUDE_TRIAL_NUMBER_FOR_PHASE,
        "note": "trial_number excluded in phase analysis to avoid phase-label leak",
    },
    "bootstrap": {
        "unit": "participant_id",
        "n_boot": N_BOOT,
        "uses_fixed_oof_predictions": True,
    },
    "fixed_folds": {
        "path": str(FIXED_FOLDS_PATH),
        "key": ["analysis", "row_index"],
        "note": "Replays GroupKFold test folds saved from the Prolific-ID metadata run.",
    },
    "analyses": results_df["analysis"].tolist(),
}

with open(manifest_path, "w") as f:
    json.dump(cp_manifest, f, indent=2)

display(results_df)


In [ ]:
# ============================================================
# Final paper plot
# ============================================================
FIG_DIR = FIGURES_DIR / "embeddings"
os.makedirs(FIG_DIR, exist_ok=True)

DPI = 600
FIGSIZE_CORRECTNESS = (3.0, 4.8)
FIGSIZE_PHASES = (4.5, 4.8)

MARKERSIZE = 9
ELINEWIDTH = 2.4
CAPSIZE = 0

LABEL_FONTSIZE = 20
XTICK_FONTSIZE = 18
YTICK_FONTSIZE = 18
LEGEND_FONTSIZE = 18

CHANCE_Y = 0.5
CHANCE_LW = 2.8

YLIM = (0.45, 0.8)
YTICKS = np.linspace(0.4, 0.8, 5)

group_labels = {1: "Same Type (Group 1)", 2: "Different Types (Group 2)"}
group_colors = {1: "red", 2: "blue"}

OFF_CORR = 0.08
OFF_PHASE = 0.13

OVERALL_COLOR = "black"
OVERALL_MS = 13
OVERALL_ALPHA = 1.0

GROUP_ALPHA_CORR = 0.45
GROUP_MS_CORR = 9

GROUP_ALPHA_PHASE = 0.90
GROUP_MS_PHASE = 9


def point_ci(ax, x, mean, lo, hi, color, *, alpha=1.0, markersize=MARKERSIZE, zorder=3):
    mean = float(mean)
    lo = float(lo)
    hi = float(hi)
    yerr = np.array([[mean - lo], [hi - mean]])
    ax.errorbar(
        x, mean, yerr=yerr,
        fmt="o",
        markersize=markersize,
        color=color,
        alpha=alpha,
        elinewidth=ELINEWIDTH,
        capsize=CAPSIZE,
        linewidth=0,
        zorder=zorder,
    )


def style_ax(ax, *, ylim=YLIM, yticks=YTICKS):
    ax.set_ylim(*ylim)
    if yticks is not None:
        ax.set_yticks(yticks)

    for side in ["top", "right", "bottom", "left"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(1.4)

    ax.tick_params(axis="x", labelsize=XTICK_FONTSIZE)
    ax.tick_params(axis="y", labelsize=YTICK_FONTSIZE)

    ax.grid(False)
    ax.axhline(CHANCE_Y, linestyle=":", color="0.55", linewidth=CHANCE_LW, zorder=0)


df = results_df.copy()
df.columns = df.columns.str.strip()

A_same = df[df["analysis"] == "B_correct_vs_incorrect__group_same"].copy()
A_diff = df[df["analysis"] == "B_correct_vs_incorrect__group_diff"].copy()
A_overall = df[df["analysis"] == "A_correct_vs_incorrect__pooled"].copy()

if len(A_same) != 1 or len(A_diff) != 1:
    raise ValueError("Expected exactly one row for each of the B_* correct_vs_incorrect group analyses.")
if len(A_overall) != 1:
    raise ValueError("Expected exactly one row for A_correct_vs_incorrect__pooled (overall).")

phase_order = [
    "D_pre_incorrect_vs_first_correct",
    "D_first_correct_vs_post_correct",
    "D_pre_incorrect_vs_post_correct",
]
phase_labels = [
    "pre →\nfirst",
    "first →\npost",
    "pre →\npost",
]

D_same = df[df["analysis"].isin([
    "D_pre_incorrect_vs_first_correct__group_same",
    "D_first_correct_vs_post_correct__group_same",
    "D_pre_incorrect_vs_post_correct__group_same",
])].copy()

D_diff = df[df["analysis"].isin([
    "D_pre_incorrect_vs_first_correct__group_diff",
    "D_first_correct_vs_post_correct__group_diff",
    "D_pre_incorrect_vs_post_correct__group_diff",
])].copy()

def _phase_key(s: str):
    s = str(s)
    for i, key in enumerate(phase_order):
        if key in s:
            return i
    return 999

D_same["__k"] = D_same["analysis"].map(_phase_key)
D_diff["__k"] = D_diff["analysis"].map(_phase_key)
D_same = D_same.sort_values("__k").drop(columns="__k")
D_diff = D_diff.sort_values("__k").drop(columns="__k")

if len(D_same) != 3 or len(D_diff) != 3:
    raise ValueError("Expected exactly 3 rows each for D_* phase contrasts for same and diff groups.")

# FIGURE 1
fig1, ax = plt.subplots(1, 1, figsize=FIGSIZE_CORRECTNESS, dpi=DPI)
style_ax(ax)

x0 = 0.0

point_ci(
    ax, x0 - OFF_CORR,
    A_same.iloc[0]["acc"], A_same.iloc[0]["acc_ci_lo"], A_same.iloc[0]["acc_ci_hi"],
    color=group_colors[1], alpha=GROUP_ALPHA_CORR, markersize=GROUP_MS_CORR, zorder=3
)
point_ci(
    ax, x0 + OFF_CORR,
    A_diff.iloc[0]["acc"], A_diff.iloc[0]["acc_ci_lo"], A_diff.iloc[0]["acc_ci_hi"],
    color=group_colors[2], alpha=GROUP_ALPHA_CORR, markersize=GROUP_MS_CORR, zorder=3
)

rO = A_overall.iloc[0]
point_ci(
    ax, x0,
    rO["acc"], rO["acc_ci_lo"], rO["acc_ci_hi"],
    color=OVERALL_COLOR, alpha=OVERALL_ALPHA, markersize=OVERALL_MS, zorder=4
)

ax.set_xlim(-0.12, 0.12)
ax.set_xticks([x0])
ax.set_xticklabels(["all correct vs\nall incorrect"])
ax.set_ylabel("accuracy", fontsize=LABEL_FONTSIZE)

handles1 = [
    plt.Line2D([0],[0], marker="o", color="black", linestyle="None",
               markersize=OVERALL_MS, alpha=1.0, label="Overall"),
    plt.Line2D([0],[0], marker="o", color=group_colors[1], linestyle="None",
               markersize=MARKERSIZE, alpha=GROUP_ALPHA_PHASE, label=group_labels[1]),
    plt.Line2D([0],[0], marker="o", color=group_colors[2], linestyle="None",
               markersize=MARKERSIZE, alpha=GROUP_ALPHA_PHASE, label=group_labels[2]),
    plt.Line2D([0],[0], linestyle=":", color="0.55", linewidth=CHANCE_LW, label="chance"),
]
fig1.legend(
    handles=handles1,
    loc="upper center",
    ncol=2,
    frameon=False,
    fontsize=LEGEND_FONTSIZE,
    bbox_to_anchor=(0.5, 1.12),
)

plt.tight_layout(rect=[0, 0, 1, 0.90])

out1 = f"fig_semantic_accuracy_correctness__compact__{time.strftime('%Y%m%d_%H%M%S')}"
fig1.savefig(f"{FIG_DIR}/{out1}.png", dpi=DPI, bbox_inches="tight", facecolor="white")
fig1.savefig(f"{FIG_DIR}/{out1}.pdf", bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig1)

print(f"Saved: {FIG_DIR}/{out1}.png and .pdf")

# FIGURE 2
fig2, ax = plt.subplots(1, 1, figsize=FIGSIZE_PHASES, dpi=DPI)
style_ax(ax)

x = np.arange(3, dtype=float)

for i in range(3):
    rS = D_same.iloc[i]
    rD = D_diff.iloc[i]

    point_ci(
        ax, x[i] - OFF_PHASE,
        rS["acc"], rS["acc_ci_lo"], rS["acc_ci_hi"],
        color=group_colors[1], alpha=GROUP_ALPHA_PHASE, markersize=GROUP_MS_PHASE, zorder=3
    )
    point_ci(
        ax, x[i] + OFF_PHASE,
        rD["acc"], rD["acc_ci_lo"], rD["acc_ci_hi"],
        color=group_colors[2], alpha=GROUP_ALPHA_PHASE, markersize=GROUP_MS_PHASE, zorder=3
    )

ax.set_xticks(x)
ax.set_xticklabels(phase_labels)
ax.set_xlim(-0.55, 2.55)
ax.set_ylabel("accuracy", fontsize=LABEL_FONTSIZE)

handles2 = [
    plt.Line2D([0],[0], marker="o", color=group_colors[1], linestyle="None",
               markersize=MARKERSIZE, alpha=GROUP_ALPHA_PHASE, label=group_labels[1]),
    plt.Line2D([0],[0], marker="o", color=group_colors[2], linestyle="None",
               markersize=MARKERSIZE, alpha=GROUP_ALPHA_PHASE, label=group_labels[2]),
    plt.Line2D([0],[0], linestyle=":", color="0.55", linewidth=CHANCE_LW, label="chance"),
]
fig2.legend(
    handles=handles2,
    loc="upper center",
    ncol=3,
    frameon=False,
    fontsize=LEGEND_FONTSIZE,
    bbox_to_anchor=(0.5, 1.12),
)

plt.tight_layout(rect=[0, 0, 1, 0.90])

out2 = f"fig_semantic_accuracy_phases__{time.strftime('%Y%m%d_%H%M%S')}"
fig2.savefig(f"{FIG_DIR}/{out2}.png", dpi=DPI, bbox_inches="tight", facecolor="white")
fig2.savefig(f"{FIG_DIR}/{out2}.pdf", bbox_inches="tight", facecolor="white")
plt.show()
plt.close(fig2)

print(f"Saved: {FIG_DIR}/{out2}.png and .pdf")

## Reasoning-Move Analyses

In [ ]:
PATCHED_PATH = DATA_DIR / "reasoning_moves" / "reasoning_moves_patched.parquet"
reasoning_moves_df = pd.read_parquet(PATCHED_PATH)

print(
    f"Loaded reasoning moves: "
    f"{len(reasoning_moves_df)} utterance rows across "
    f"{reasoning_moves_df['participant_id'].astype(str).nunique()} participants."
)

In [ ]:
N_BOOT = 10000
SEED = 0

PID_COL = "participant_id"
TRIAL_COL = "trial_number"
GROUP_COL = "group"
PHASE_COL = "phase"

G1 = 1
G2 = 2

PHASE_ORDER = ["pre_incorrect", "first_correct", "post_correct"]
PHASE_PAIRS = [
    (PHASE_ORDER[0], PHASE_ORDER[1], "pre→first"),
    (PHASE_ORDER[1], PHASE_ORDER[2], "first→post"),
    (PHASE_ORDER[0], PHASE_ORDER[2], "pre→post"),
]

DISPLAY_NAME = {
    "meta_strategy": "Meta Strategy & Other",
}

df = reasoning_moves_df.copy()
label_col = "move_label" if "move_label" in df.columns else "label"

df = df[df[label_col].notna()].copy()
df = df[df[label_col] != "missing"].copy()

req = [PID_COL, TRIAL_COL, GROUP_COL, PHASE_COL, label_col]
missing = [c for c in req if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df[TRIAL_COL] = pd.to_numeric(df[TRIAL_COL], errors="coerce").astype("Int64")
df[GROUP_COL] = pd.to_numeric(df[GROUP_COL], errors="coerce").astype("Int64")
df[label_col] = df[label_col].astype(str)
df[PHASE_COL] = df[PHASE_COL].astype(str)

df = df[df[PHASE_COL].notna()].copy()
df = df[df[PHASE_COL] != "unlabeled"].copy()
df = df[df[PHASE_COL].isin(PHASE_ORDER)].copy()
df[PHASE_COL] = pd.Categorical(df[PHASE_COL], categories=PHASE_ORDER, ordered=True)

stack_order = [
    "proposal",
    "evaluation",
    "restatement",
    "filler",
    "affect",
    "meta_strategy",
    "categorization",
]
labels_present = set(df[label_col].unique())
labels = [l for l in stack_order if l in labels_present]

trial_counts = (
    df.groupby([PID_COL, TRIAL_COL, GROUP_COL, PHASE_COL, label_col], observed=True)
      .size().rename("count").reset_index()
)

trial_totals = (
    df.groupby([PID_COL, TRIAL_COL, GROUP_COL, PHASE_COL], observed=True)
      .size().rename("total").reset_index()
)

all_trials = trial_totals[[PID_COL, TRIAL_COL, GROUP_COL, PHASE_COL]].copy()
all_labels = pd.DataFrame({label_col: pd.Series(labels, dtype="object")})
grid = all_trials.merge(all_labels, how="cross")

trial_props = (
    grid.merge(
        trial_counts,
        on=[PID_COL, TRIAL_COL, GROUP_COL, PHASE_COL, label_col],
        how="left",
        validate="many_to_one"
    )
    .merge(
        trial_totals,
        on=[PID_COL, TRIAL_COL, GROUP_COL, PHASE_COL],
        how="left",
        validate="many_to_one"
    )
)

trial_props["count"] = trial_props["count"].fillna(0.0)
trial_props["prop"] = trial_props["count"] / trial_props["total"]

pid_props = (
    trial_props.groupby([PID_COL, GROUP_COL, PHASE_COL, label_col], observed=True)["prop"]
    .mean()
    .reset_index()
)

def paired_pid_deltas(pid_props_sub: pd.DataFrame, phase_a: str, phase_b: str) -> np.ndarray:
    wide = (
        pid_props_sub[pid_props_sub[PHASE_COL].isin([phase_a, phase_b])]
        .pivot_table(index=PID_COL, columns=PHASE_COL, values="prop", aggfunc="mean", observed=True)
    )
    if phase_a not in wide.columns or phase_b not in wide.columns:
        return np.array([], dtype=float)
    wide = wide[[phase_a, phase_b]].dropna()
    d = (wide[phase_b] - wide[phase_a]).to_numpy(dtype=float)
    return d[np.isfinite(d)]

def bootstrap_ci95_mean(x: np.ndarray, *, n_boot: int, seed: int):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n == 0:
        return np.nan, np.nan, np.nan, 0
    point = float(x.mean())
    if n == 1:
        return point, np.nan, np.nan, 1

    rng = np.random.default_rng(seed)
    boots = rng.choice(x, size=(n_boot, n), replace=True).mean(axis=1)
    lo, hi = np.quantile(boots, [0.025, 0.975])
    return point, float(lo), float(hi), n

rows = []
for g in [G1, G2]:
    for lab in labels:
        sub = pid_props[(pid_props[GROUP_COL] == g) & (pid_props[label_col] == lab)].copy()
        for (ph_a, ph_b, name) in PHASE_PAIRS:
            deltas = paired_pid_deltas(sub, ph_a, ph_b)

            cell_seed = (
                SEED
                + 10_000 * int(g)
                + 1_000 * labels.index(lab)
                + 37 * PHASE_ORDER.index(ph_a)
                + 53 * PHASE_ORDER.index(ph_b)
            )

            point, lo, hi, n = bootstrap_ci95_mean(deltas, n_boot=N_BOOT, seed=cell_seed)
            sig = bool(np.isfinite(lo) and np.isfinite(hi) and (lo > 0 or hi < 0))

            rows.append({
                "group": int(g),
                "label": lab,
                "contrast": name,
                "phase_a": ph_a,
                "phase_b": ph_b,
                "delta_point": point,
                "ci_lo": lo,
                "ci_hi": hi,
                "sig_ci_excludes_0": sig,
                "n_paired_participants": int(n),
            })

within_df = pd.DataFrame(rows)
within_df["label"] = pd.Categorical(within_df["label"], categories=labels, ordered=True)
within_df["contrast"] = pd.Categorical(
    within_df["contrast"],
    categories=[c[2] for c in PHASE_PAIRS],
    ordered=True,
)
within_df = within_df.sort_values(["group", "contrast", "label"]).reset_index(drop=True)

group_names = {G1: "Same Type (Group 1)", G2: "Different-Types (Group 2)"}

categorization_color = "#FFD500"
default_colors = sns.color_palette("pastel", max(len(labels) - 1, 1))

label_colors = {}
if len(labels) > 1:
    label_colors = {lbl: col for lbl, col in zip(labels[:-1], default_colors[:len(labels)-1])}
if "categorization" in labels:
    label_colors["categorization"] = categorization_color
elif len(labels) == 1:
    label_colors[labels[0]] = default_colors[0]

In [ ]:
# -----------------------------
# Output
# -----------------------------
OUT_DIR = FIGURES_DIR / "reasoning" / "by_phase"
os.makedirs(OUT_DIR, exist_ok=True)

FIGSIZE = (7.8, 4.8)
DPI = 600

TITLE_FONTSIZE  = 18
LABEL_FONTSIZE  = 16
TICK_FONTSIZE   = 14
LEGEND_FONTSIZE = 8

POINT_ALPHA   = 0.9
POINT_MS_BASE = 8
POINT_MS_SIG  = 8

ERR_LW_BASE = 3.0
ERR_LW_SIG  = 3.0
CAPSIZE = 0 

TICK_LEN_MAJOR = 6
TICK_W_MAJOR   = 1.5
TICK_LEN_MINOR = 3
TICK_W_MINOR   = 1.0

YTICK_STEP = 0.10
YFMT = "%.2f"

offsets = {"pre→first": -0.22, "first→post": 0.0, "pre→post": 0.22}
markers = {"pre→first": "o", "first→post": "s", "pre→post": "^"}

contrast_names = [c[2] for c in PHASE_PAIRS]

g = G1
sub = (
    within_df[within_df["group"] == g]
    .copy()
    .set_index(["label", "contrast"])
)

fig, ax = plt.subplots(1, 1, figsize=FIGSIZE, dpi=DPI)
ax.axhline(0.0, linestyle=":", linewidth=2.4, color="0.55", zorder=0)

x = np.arange(len(labels))

for contrast in contrast_names:
    for i, lab in enumerate(labels):
        if (lab, contrast) not in sub.index:
            continue

        r = sub.loc[(lab, contrast)]
        y  = float(r["delta_point"])
        lo = float(r["ci_lo"])
        hi = float(r["ci_hi"])
        sig = bool(r.get("sig_ci_excludes_0", False))

        xi = x[i] + offsets.get(contrast, 0.0)
        c = label_colors.get(lab, "black")

        ax.plot(
            xi, y,
            marker=markers.get(contrast, "o"),
            markersize=POINT_MS_SIG if sig else POINT_MS_BASE,
            linestyle="none",
            color=c,
            alpha=POINT_ALPHA,
            zorder=3,
        )

        if np.isfinite(lo) and np.isfinite(hi):
            ax.errorbar(
                xi, y,
                yerr=[[y - lo], [hi - y]],
                fmt="none",
                ecolor=c,
                elinewidth=ERR_LW_SIG if sig else ERR_LW_BASE,
                capsize=CAPSIZE,
                alpha=POINT_ALPHA,
                zorder=2,
            )

        if sig and np.isfinite(y):
            ax.text(
                xi, y + 0.01,
                "★",
                ha="center", va="bottom",
                fontsize=12, color="black",
                zorder=4,
            )

ax.set_title(group_names.get(g, "Same Type (Group 1)"), fontsize=TITLE_FONTSIZE, pad=10)

ax.set_xticks(x)
ax.set_xticklabels(
    [DISPLAY_NAME.get(l, l.replace("_", " ").title()) for l in labels],
    rotation=30,
    ha="right",
    fontsize=TICK_FONTSIZE,
)

ax.set_ylabel("Δ proportion\n(phase b − phase a)", fontsize=LABEL_FONTSIZE)

y_lo = np.nanmin([sub["ci_lo"].min(), sub["delta_point"].min()])
y_hi = np.nanmax([sub["ci_hi"].max(), sub["delta_point"].max()])
ax.set_ylim(y_lo - 0.03, y_hi + 0.03)

ymin, ymax = ax.get_ylim()
yticks = np.arange(
    np.floor(ymin / YTICK_STEP) * YTICK_STEP,
    np.ceil(ymax / YTICK_STEP) * YTICK_STEP + 1e-9,
    YTICK_STEP
)
ax.set_yticks(yticks)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter(YFMT))

ax.tick_params(
    axis="both",
    which="major",
    labelsize=TICK_FONTSIZE,
    length=TICK_LEN_MAJOR,
    width=TICK_W_MAJOR,
    direction="out",
)

ax.minorticks_on()
ax.tick_params(
    axis="both",
    which="minor",
    length=TICK_LEN_MINOR,
    width=TICK_W_MINOR,
    direction="out",
)

ax.grid(False)

handles = [
    plt.Line2D([0], [0], marker=markers[name], color="black",
               linestyle="none", markersize=9)
    for name in contrast_names
]
ax.legend(
    handles,
    contrast_names,
    title="Phase Transition",
    frameon=False,
    fontsize=LEGEND_FONTSIZE,
    title_fontsize=LEGEND_FONTSIZE,
    loc="upper right",
)

plt.tight_layout()

out_prefix = "phase_proportions_same_group"
fig.savefig(f"{OUT_DIR}/{out_prefix}.png", dpi=DPI, bbox_inches="tight", facecolor="white")
fig.savefig(f"{OUT_DIR}/{out_prefix}.pdf", dpi=DPI, bbox_inches="tight", facecolor="white")
plt.show()

print(f"Saved: {OUT_DIR}/{out_prefix}.png and .pdf")


In [ ]:
# ============================================================
# Phase means for reasoning-move proportions
# ============================================================
phase_rows = []
for (g, ph, lab), sub in pid_props.groupby([GROUP_COL, PHASE_COL, label_col], observed=True):
    cell_seed = (
        SEED
        + 100_000 * int(g)
        + 1_000 * PHASE_ORDER.index(str(ph))
        + 37 * labels.index(str(lab))
    )
    m, lo, hi, n = bootstrap_ci95_mean(sub["prop"].to_numpy(), n_boot=N_BOOT, seed=cell_seed)
    phase_rows.append({
        "group": int(g),
        "phase": str(ph),
        "label": str(lab),
        "mean": m,
        "ci_lo": lo,
        "ci_hi": hi,
        "n_participants": int(n),
    })

phase_summary_df = pd.DataFrame(phase_rows)
phase_summary_df["group"] = phase_summary_df["group"].astype(int)
phase_summary_df["phase"] = pd.Categorical(phase_summary_df["phase"], categories=PHASE_ORDER, ordered=True)
phase_summary_df["label"] = pd.Categorical(phase_summary_df["label"], categories=labels, ordered=True)
phase_summary_df = phase_summary_df.sort_values(["group", "label", "phase"]).reset_index(drop=True)

display(phase_summary_df.head(10))

In [ ]:
# ============================================================
# Distinct candidate moves per trial
# Uses unique proposal trace_id counts
# ============================================================
TRACE_COL = "trace_id"

proposal_trial_counts = (
    df[df[label_col].eq("proposal") & df[TRACE_COL].notna()]
      .groupby([PID_COL, TRIAL_COL, GROUP_COL, PHASE_COL], observed=True)[TRACE_COL]
      .nunique()
      .rename("n_unique_proposals")
      .reset_index()
)

all_trials_phase = (
    df[[PID_COL, TRIAL_COL, GROUP_COL, PHASE_COL]]
      .drop_duplicates()
      .copy()
)

proposal_trial_counts = all_trials_phase.merge(
    proposal_trial_counts,
    on=[PID_COL, TRIAL_COL, GROUP_COL, PHASE_COL],
    how="left",
    validate="one_to_one",
)
proposal_trial_counts["n_unique_proposals"] = proposal_trial_counts["n_unique_proposals"].fillna(0.0)

pid_propcount = (
    proposal_trial_counts
    .groupby([PID_COL, GROUP_COL, PHASE_COL], observed=True)["n_unique_proposals"]
    .mean()
    .reset_index()
    .rename(columns={"n_unique_proposals": "mean_unique_proposals_per_trial"})
)

def paired_pid_deltas_counts(pid_df: pd.DataFrame, phase_a: str, phase_b: str) -> np.ndarray:
    wide = (
        pid_df[pid_df[PHASE_COL].isin([phase_a, phase_b])]
        .pivot_table(
            index=PID_COL,
            columns=PHASE_COL,
            values="mean_unique_proposals_per_trial",
            aggfunc="mean",
            observed=True,
        )
    )
    if phase_a not in wide.columns or phase_b not in wide.columns:
        return np.array([], dtype=float)
    wide = wide[[phase_a, phase_b]].dropna()
    d = (wide[phase_b] - wide[phase_a]).to_numpy(dtype=float)
    return d[np.isfinite(d)]

rows_counts = []
for g in [G1, G2]:
    pid_g = pid_propcount[pid_propcount[GROUP_COL].eq(g)].copy()
    for (ph_a, ph_b, name) in PHASE_PAIRS:
        deltas = paired_pid_deltas_counts(pid_g, ph_a, ph_b)

        cell_seed = (
            SEED
            + 99_000 * int(g)
            + 101 * PHASE_ORDER.index(ph_a)
            + 103 * PHASE_ORDER.index(ph_b)
        )

        point, lo, hi, n = bootstrap_ci95_mean(deltas, n_boot=N_BOOT, seed=cell_seed)
        sig = bool(np.isfinite(lo) and np.isfinite(hi) and (lo > 0 or hi < 0))

        rows_counts.append({
            "group": int(g),
            "contrast": name,
            "phase_a": ph_a,
            "phase_b": ph_b,
            "delta_point": point,
            "ci_lo": lo,
            "ci_hi": hi,
            "sig_ci_excludes_0": sig,
            "n_paired_participants": int(n),
        })

within_counts_df = pd.DataFrame(rows_counts)
within_counts_df["contrast"] = pd.Categorical(
    within_counts_df["contrast"],
    categories=[c[2] for c in PHASE_PAIRS],
    ordered=True,
)
within_counts_df = within_counts_df.sort_values(["group", "contrast"]).reset_index(drop=True)

display(within_counts_df)


In [ ]:
within_df[
    within_df["label"].isin(["categorization", "proposal"])
][[
    "group", "label", "contrast", "delta_point", "ci_lo", "ci_hi", "sig_ci_excludes_0"
]].sort_values(["label", "group", "contrast"])


In [ ]:
phase_summary_df[
    phase_summary_df["label"] == "categorization"
][[
    "group", "phase", "mean", "ci_lo", "ci_hi", "n_participants"
]].sort_values(["group", "phase"])
